# Testing your best model configurations
This notebook lets you train a new model on the entirety of your training data, and then validate its performance on the test partition. 

### How to use:
Put the directory of your `best_model` folder in the config cell below and run the notebook. The output will be placed in this folder too.

In [ ]:
BEST_RUN_DIR = r"output\13-06-2026--11-33\job_0\best_model"

DEVICE = "cuda"

ADDITIONAL_EPOCHS = 0

From here on, you do not need to alter the code.

In [ ]:
import logging
import os
import sys
import torch
import yaml

from functools import partial
from logging.handlers import RotatingFileHandler
from jsonschema import validate, ValidationError
from bcos.modules.losses import BinaryCrossEntropyLoss


from augment import SpecAugment, AudioAugment
from train import train, evaluate, METRICS
from data import to_dataloaders
from esc_audio_dataset import ESCAudioDataset, LABEL_MAP
from bcos.modules.bcosconv2d import BcosConv2d
from config.config_validation_template import CONFIG_TEMPLATE
from custom_logger_formatter import CustomLoggerFormatter
from main import DATASET_MAPPING
from resnet_bcos import make_resnet18, make_resnet34, make_resnet50
from resnet_18_baseline import BaselineResNet
from tune import TUNABLE_PARAMS
from visualise import visualise_training, plot_confusion_matrix

In [ ]:
# Initialise Logger.

def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("test logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
f_ch = RotatingFileHandler(f"{BEST_RUN_DIR}/testing.log")
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the testing process.")

In [ ]:
# validate the provided config file.
with open(f"{BEST_RUN_DIR}/run_config.yml", 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(f"{BEST_RUN_DIR}/../../config.yml", 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

run = CONFIG["jobs"]["job0"]
for tunable_param in TUNABLE_PARAMS.keys():
    if tunable_param == "small_inputs":
        run["model_params"][tunable_param] = run["model_params"][tunable_param][0]
    else:
        run[tunable_param] = run[tunable_param][0]

run["n_epochs"] += ADDITIONAL_EPOCHS

logger.info(f"config: {CONFIG}")

In [ ]:
# Load the data
train_dataset = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["train_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=run["sample_rate"],
    duration=run["duration"],
    n_fft=run["n_fft"],
    hop_length=run["hop_length"],
    n_mels=run["n_mels"],
    top_db=run["top_db"],
    pre_transform=AudioAugment(),
    post_transform=SpecAugment()
)

test_data = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["test_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=run["sample_rate"],
    duration=run["duration"],
    n_fft=run["n_fft"],
    hop_length=run["hop_length"],
    n_mels=run["n_mels"],
    top_db=run["top_db"],
)

# Normalise
logger.debug("Fitting normalisation.")
train_dataset.fit_normalisation(list(range(len(train_dataset))))
test_data.mean = train_dataset.mean
test_data.std = train_dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean={train_dataset.mean}, std={train_dataset.std}"
)

# Val dataset only needed to re-use existing code, not actually used
val_dataset = torch.utils.data.Subset(train_dataset, list(range(0, run["batch_size"]*5)))
logger.debug(f"{len(train_dataset) = }, {len(val_dataset) = }")

# Convert to dataloaders
logger.debug("converting to dataloaders")
train_dataloader, val_dataloader = to_dataloaders(
            [train_dataset, val_dataset], 
            batch_sizes=[run["batch_size"]] * 2, 
            shuffles=[True, False],
            logger=logger,
            num_workers=CONFIG["general"]["num_data_workers"],
            pin_memory=True, # TODO: check if this should be replaced with run["lazy"]
            persistent_workers=True if CONFIG["general"]["num_data_workers"] > 0 else False,
        )

test_dataloader = to_dataloaders(
    [test_data], 
    batch_sizes=[run["batch_size"]], 
    shuffles=[False],
    logger=logger,
    num_workers=0,
    pin_memory=False,
)[0]


In [ ]:
# Process the model
logger.debug(f"Initialising the model ({run['model']})")
models = {
    "resnet18_bcos": (
        make_resnet18, {
            "logger": logger,
            "num_classes": train_dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": run["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=run["b"], max_out=2),
        }
    ),
    "resnet34_bcos": (
        make_resnet34, {
            "logger": logger,
            "num_classes": train_dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": run["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=run["b"], max_out=2),
        }
    ),
    "resnet50_bcos": (
        make_resnet50, {
            "logger": logger,
            "num_classes": train_dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": run["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=run["b"], max_out=2),
        }
    ),
    "resnet18_baseline": (
        lambda **kwargs: BaselineResNet(kwargs["num_classes"], kwargs["logger"]),
        {"logger": logger, "num_classes": train_dataset.get_n_classes()}
    )   
}
model = None
for name, (cls, kwargs) in models.items():
    if run['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)

# model requirements
logger.debug(f"Initialising the optimiser ({run['optimiser']})")
optimisers = {
    "adam": (torch.optim.Adam, {
        "params": model.parameters(),
        "lr": run["learning_rate"],
        "weight_decay": run["weight_decay"]
    })
}
OPTIMISER = None
for name, (cls, kwargs) in optimisers.items():
    if run['optimiser'].lower() in name:
        OPTIMISER = cls(**kwargs)
        break
assert OPTIMISER is not None, \
    "Provided optimiser in config does not exist."

LOSS_FN = BinaryCrossEntropyLoss()

arguments = {
    "model" : model,
    "loss_fn" : LOSS_FN,
    "optimiser": OPTIMISER,
    "n_epochs" : run["n_epochs"],
    "device" : DEVICE,
    "logger" : logger,
    "pruning_callback": None
}

In [ ]:
train_losses, train_metrics, val_losses, val_metrics, model = train(
    train_dataloader=train_dataloader, 
    val_dataloader=val_dataloader,
    **arguments
)
train_losses_std, train_metrics_std = None, None
val_losses_std, val_metrics_std = None, None

model.save(BEST_RUN_DIR, prefix="fully_trained_")


In [ ]:
visualise_training(
    train_losses, 
    train_metrics,
    val_losses, 
    val_metrics,
    BEST_RUN_DIR + "/fully_trained_",
)

In [ ]:
train_accuracy, train_predictions, train_targets = evaluate(
    dataloader=train_dataloader, 
    model=model,
    device=DEVICE,
    logger=logger,
)
logger.critical(f"Train accuracy: {train_accuracy}")

plot_confusion_matrix(
    train_targets,
    train_predictions,
    LABEL_MAP.keys(),
    None,
    "train",
    BEST_RUN_DIR+"/",
    logger
)

In [ ]:
logger.info("Evaluating on test set.")
test_accuracy, test_predictions, test_targets = evaluate(
    dataloader=test_dataloader, 
    model=model,
    device=DEVICE,
    logger=logger,
)
logger.critical(f"Test accuracy: {test_accuracy}")

plot_confusion_matrix(
    test_targets,
    test_predictions,
    LABEL_MAP.keys(),
    None,
    "test",
    BEST_RUN_DIR+"/",
    logger
)

In [ ]:
with open(f"{BEST_RUN_DIR}\\metrics.md", "w") as file:
    file.write("# Testing metrics\n")
    file.write("training (on entire dataset):\n")
    file.write(f"accuracy: {train_accuracy}\n")
    file.write("testing:\n")
    file.write(f"accuracy: {test_accuracy}\n")